# contiguous-layout — ex1: predict-then-verify the strides of a 3-D contiguous tensor

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `contiguous-layout`. Running the final beacon cell reports progress against the `PyTorch: Contiguous layout` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Contiguous layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`contiguous-layout`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "contiguous-layout"
DD_SUBTOPIC = "PyTorch: Contiguous layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Contiguous layout — quick refresher

A tensor is **contiguous** when its in-memory layout is row-major: the last axis has stride 1 and each earlier axis's stride equals the product of all sizes to its right. `x.is_contiguous()` reports the answer; `x.stride()` lets you check by hand.

**Why it matters.**
- `view()` requires contiguous input — call `.contiguous()` first if you've transposed/permuted/strided into a non-contiguous layout.
- `reshape()` will silently copy when needed; `view()` will not.
- Many low-level kernels (cuDNN convs, `as_strided`) read raw stride values — passing them a tensor whose strides you didn't expect is the single biggest source of off-by-`H*W` bugs in CNN-from-scratch code.

**Useful identities for a contiguous `(d0, d1, ..., dN)` tensor:**
- `stride(N) == 1`
- `stride(k) == d[k+1] * d[k+2] * ... * d[N]`

### Exercise 1 — predict-then-verify the strides of a 3-D contiguous tensor

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Compute the strides of a contiguous N-dim tensor from its shape using the row-major formula, and verify against `.stride()` and `.is_contiguous()`.
> Keywords: strides, row-major, is_contiguous
> ```

**KCs targeted:** `contiguous-stride-formula`, `is-contiguous-check`

Implement `ex1_predicted_strides(shape)`. Given a tuple of sizes `shape = (d0, d1, ..., dN)`, return the tuple of strides a contiguous (row-major) tensor of that shape would have.

**Formula.** For a contiguous tensor:
- `stride[-1] = 1`
- `stride[k] = d[k+1] * d[k+2] * ... * d[N]` for all `k < N`

Inputs: `shape` — tuple of `int >= 1`.
Output: tuple of `int`, same length as `shape`.

**No torch in this function — pure Python math.** The test will use a real `t.zeros(shape)` to confirm your formula matches what PyTorch actually allocates.

In [ ]:
def ex1_predicted_strides(shape: tuple) -> tuple:
    strides = []
    running = 1
    for dim in reversed(shape):
        strides.append(running)
        running *= dim
    return tuple(reversed(strides))


<details><summary>Solution</summary>

```python
def ex1_predicted_strides(shape: tuple) -> tuple:
    strides = []
    running = 1
    for dim in reversed(shape):
        strides.append(running)
        running *= dim
    return tuple(reversed(strides))
```

**The walk goes right-to-left.** The last axis always has stride 1 (you advance by one element to step along it). Each earlier axis multiplies in the size of everything to its right.

**Why this is a load-bearing skill for CNN-from-scratch.** Building `as_strided` windows for `conv1d_minimal` requires you to KNOW these strides — you'll pull them off `x.stride()` rather than recompute, but if your mental model is wrong you won't catch the bug when a transposed input arrives with non-contiguous strides.

**Gotcha — size-1 axes.** A `(2, 1, 3, 4)` contiguous tensor has stride `(12, 12, 4, 1)`. The size-1 axis's stride equals the stride of the axis to its left because there's nothing to advance over.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()